<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">پیشنهاد، اجرا و پایان را جدا بشمارید</h1>
<p style="text-align:right">درس 86 از 92 · چه کسی تصمیم می‌گیرد یک گام دیگر اجرا شود؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">80-controller</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-14/chapter-01/80-controller.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right"><bdi dir="ltr">Trace</bdi> یک <bdi dir="ltr">Controller</bdi> واقعی را بخوانید و ثابت کنید تعداد پیشنهادها با ابزارهای اجراشده یکسان نیست.</p><p style="text-align:right">پیش‌نیاز: <bdi dir="ltr">Context</bdi>، شاهد، <bdi dir="ltr">Memory</bdi>، <bdi dir="ltr">Schema</bdi> ابزار و <bdi dir="ltr">Verifier</bdi>؛ <bdi dir="ltr">Fixture</bdi> ازپیش‌نوشته‌شده را با مدل آموزش‌دیده یکی نگیرید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۹۰–۱۵۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر سقف ابزار صفر باشد، یک پیشنهاد معتبر باید چند اجرای واقعی ایجاد کند؟ اگر پاسخ نهایی بدون <bdi dir="ltr">Verifier</bdi> برسد، <bdi dir="ltr">verified</bdi> باید چه باشد؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
torch.set_num_threads(1)
torch.manual_seed(41)
from mini_gpt.assistant import ScriptedFixture, MiniGPTBackend, run_assistant, PROTOCOL
from mini_gpt.context import ContextItem,build_context
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
from mini_gpt.tokenizer import CharacterTokenizer
from mini_gpt.retrieval import COURSE_DOCUMENTS,chunk_document,keyword_search
from mini_gpt.memory import MemoryStore,MemoryRecord
calls = [{'action':'tool','name':'multiply','arguments':{'a':25,'b':48}},
         {'action':'finish','answer':'1200','citations':[]}]
good = run_assistant('25 * 48?',ScriptedFixture(calls),
                     verify_answer=lambda answer,citations,tools: answer == '1200')
blocked = run_assistant('25 * 48?',ScriptedFixture(calls),max_tool_calls=0)
print(good.backend,good.status,blocked.status)
# Full controller prompt, random weights, real generation, NO fallback.
question = '25*48?'
pack = build_context(question,[ContextItem('controller:protocol',PROTOCOL,'instruction')],
                     max_tokens=4096,reserve_tokens=4,count_tokens=len)
tokenizer = CharacterTokenizer.from_text(pack.prompt)
model = MiniGPT(ModelConfig(tokenizer.vocab_size,len(pack.prompt)+8,8,2,1,0.))
real = MiniGPTBackend(model,tokenizer,max_new_tokens=4)
real_run = run_assistant(question,real)
raw = next(event['text'] for event in real_run.events if event['state']=='propose')
print(real.label,'UNTRAINED raw output:',repr(raw),'status:',real_run.status)
print('real model counts:',real.last_counts)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">audit_run(result)</code> دیکشنری <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">proposals/executions/verified/safe_to_report</code> بدهد. دو شمار اول از رویدادهای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">propose/execute</code> در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">result.events</code> بیایند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">verified</code> را از خود نتیجه بخوانید و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">safe_to_report</code> فقط وقتی <bdi dir="ltr">True</bdi> باشد که هم وضعیت <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">finished</code> و هم <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">verified=True</code> است. این معیار آموزشیِ گزارش پاسخِ بررسی‌شده است، نه تضمین جامع ایمنی.</p>
</div>

In [ ]:
def audit_run(result):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = audit_run(good)
    if result is None: return False
    assert result == {'proposals':2,'executions':1,'verified':True,'safe_to_report':True}
    assert audit_run(blocked) == {'proposals':1,'executions':0,'verified':False,'safe_to_report':False}
    unchecked = run_assistant('answer?',ScriptedFixture([{'action':'finish','answer':'7','citations':[]}]))
    assert unchecked.status == 'finished'
    assert audit_run(unchecked) == {'proposals':1,'executions':0,'verified':False,'safe_to_report':False}
    rejected = run_assistant('answer?',ScriptedFixture([{'action':'finish','answer':'7','citations':[]}]),
                             verify_answer=lambda a,c,t: False)
    assert audit_run(rejected)['safe_to_report'] is False
    assert real.calls == 1 and real.last_counts['generated_tokens'] == 4
    assert real_run.status == 'invalid_action' and not real_run.tool_results
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط انتخاب صریح یک کلید حافظه را در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">memory_keys</code> تغییر دهید؛ خود <bdi dir="ltr">Store</bdi>، سندها، سؤال و پاسخ <bdi dir="ltr">Fixture</bdi> ثابت بمانند. شناسه‌های واردشده به <bdi dir="ltr">Context</bdi> را مقایسه کنید. بدون انتخاب کلید، هیچ رکوردی وارد نمی‌شود. پاسخ <bdi dir="ltr">Fixture</bdi> عمدی ثابت است: این آزمایش رسیدن دادهٔ مجاز به سامانه را نشان می‌دهد، نه استفادهٔ هوشمندانهٔ مدل از حافظه.</p>
</div>

In [ ]:
chunks = [c for document in COURSE_DOCUMENTS for c in chunk_document(document)]
hit = keyword_search('Checkpoint',chunks,k=1)[0]
memory = MemoryStore([MemoryRecord('study-unit','minutes','user explicitly stated the unit')])
response = {'action':'finish','answer':hit.chunk.text,'citations':[hit.chunk.id]}
for keys in ((),('study-unit',)):
    result = run_assistant('Checkpoint',ScriptedFixture([response]),chunks=chunks,memory=memory,memory_keys=keys,
                           verify_answer=lambda a,c,t: a == hit.chunk.text and c == (hit.chunk.id,))
    print('selected memory keys, status, included:',keys,result.status,result.context_ids)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">شرط خراب با <bdi dir="ltr">OR</bdi> اجازهٔ ابزار می‌دهد، حتی وقتی یکی از دو بودجه تمام شده است. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">can_execute(step_index,call_count,max_steps,max_calls)</code> برای چهار عدد صحیح نامنفی بنویسید: اجرای ابزار فقط پیش از رسیدن هر دو شمارنده به سقف مجاز است. این شرط مربوط به ابزار است؛ پاسخ نهاییِ بدون ابزار را ممنوع نمی‌کند.</p>
</div>

In [ ]:
step_index,call_count,max_steps,max_calls = 0,2,4,2
print('wrong OR permission:',step_index < max_steps or call_count < max_calls)
repeated = run_assistant('25 * 48?',ScriptedFixture([calls[0],calls[0]]))
print('same tool call repeated:',repeated.status,'executed:',len(repeated.tool_results))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def can_execute(step_index, call_count, max_steps, max_calls):
    # TODO
    return None

In [ ]:
def test_repair():
    result = can_execute(0,0,4,2)
    if result is None: return False
    assert result is True
    for state in ((0,2,4,2),(4,0,4,2),(4,2,4,2),(0,0,1,0),(0,0,0,1)):
        assert can_execute(*state) is False
    assert can_execute(3,1,4,2) is True
    assert repeated.status == 'repeated_action' and len(repeated.tool_results) == 1
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">assistant.py</code> واقعاً <bdi dir="ltr">Context</bdi>، <bdi dir="ltr">Retrieval</bdi>، <bdi dir="ltr">Memory</bdi> و ابزارها را کنار هم می‌گذارد. در این دفتر هم مسیر <bdi dir="ltr">Fixture</bdi> و هم تولید <bdi dir="ltr">Token</bdi> از <bdi dir="ltr">MiniGPT</bdi> اجرا شد، اما فقط اولی سناریوی دست‌نویس ابزار را دنبال کرد. گزارش این دو را ادغام نکنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">کدام نتیجه دربارهٔ درستی <bdi dir="ltr">Controller</bdi> بود، کدام دربارهٔ اتصال واقعی مدل و کدام قابلیت هنوز با وزن‌های تصادفی ثابت نشده است؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-14/chapter-01/80-controller.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/80-controller.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>